In [1]:
import sys
import os

# Add project root to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
from utils.remove_correlated_features import remove_correlated_features
from utils.feature_engineering.lagged_features import get_data_lagged
from utils.perform_xgboost_selection import perform_xgboost_selection
from utils.feature_engineering.ewma_features import get_data_ewma
from utils.feature_engineering.hybrid_features import get_data_hybrid
from utils.feature_engineering.variance_inclusion import get_data_hybrid_w_var


# Feature Selection 

The aim is to create a model for prodicting points for players in fpl. Since each position gets points based on different parameters, it is decided that one regression model is to be implemented for each position (GK, DEF, MID, FWD).

The goal of this analysis is to decide on what features are relevant for each position, this in order to reduce the compute needed to create models (I run everything locally). 

Initial plan of using correlation metrict to perform feature selection is to fragile. Using XGBoost instead based on this article; 
https://medium.com/@dhanyahari07/feature-selection-using-xgboost-f0622fb70c4d

Training and feature selection is performed on data from 22/23 and 23/24 season. The final evaluation is performed on data from the 24/25 season.


##### On using XGBoost as a feature selection pipeline
XGBoost calculates three types of feature importance scores:

* Gain: Average loss reduction gained when using a feature for splitting.

* Cover: The number of times a feature is used to split data across trees weighted by training data points.

* Weight: Total number of times a feature is used to split data across all trees.

#### Feature Selection Hyperparameters

In [6]:
TARGET = 'total_points'  
VARIANCE_TRESHOLD: float = 0.015     # Minimum variance for a feature to be kept
CORR_FEATURE_TRESHOLD: float = 0.75  # Maximum correlation allowed between two features
PRINT_INFO: bool = True

#### Relevant position featurs

In [4]:
FIELD_PLAYER_RELEVANT_FEATURES = [
                'name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 
                'creativity', 'expected_assists', 'expected_goal_involvements', 
                'expected_goals','expected_goals_conceded', 'goals_scored', 'ict_index', 'influence', 'minutes', 
                'opponent_team', 'own_goals', 'penalties_missed', 'red_cards', 
                'selected', 'team_a_score', 'team_h_score', 'threat', 'total_points', 
                'transfers_balance', 'value', 'was_home', 
                'yellow_cards', 'GW'
                ]

GOALKEEPER_RELEVANT_FEATURES = [
                'name', 'position', 'GW', 'xP', 'bonus', 'bps', 'clean_sheets',  
                'expected_goals_conceded', 'ict_index', 'influence', 'minutes', 
                'opponent_team', 'own_goals', 'red_cards', 'saves',
                'selected', 'team_a_score', 'team_h_score', 'total_points', 
                'transfers_balance', 'value', 'was_home', 
                'yellow_cards', 
                ]

### Getting data and performing selection

### FWD

#### Initial test
Using lagged features vs ewma

#### Lagged features

In [4]:
# Get data for FWD players
warnings.filterwarnings("ignore", category=UserWarning)

fwd_data = get_data_lagged('FWD', FIELD_PLAYER_RELEVANT_FEATURES)
fwd_data = remove_correlated_features(fwd_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features_lagged.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

Found 71 pairs of highly correlated features.
Removed 39 features due to high correlation.
Original number of features; 130. Final number of features: 91
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 89 different feature thresholds...
Thresh=0.0000, n=89, MSE=11.1225, Best Score=11.1225
Thresh=0.0000, n=89, MSE=11.1225, Best Score=11.1225
Thresh=0.0000, n=89, MSE=11.1225, Best Score=11.1225
Thresh=0.0000, n=89, MSE=11.1225, Best Score=11.1225
Thresh=0.0000, n=89, MSE=11.1225, Best Score=11.1225
Thresh=0.0000, n=89, MSE=11.1225, Best Score=11.1225
Thresh=0.0000, n=89, MSE=11.1225, Best Score=11.1225
Thresh=0.0000, n=89, MSE=11.1225, Best Score=11.1225
Thresh=0.0022, n=81, MSE=11.1225, Best Score=11.1225
Thresh=0.0033, n=80, MSE=11.1162, Best Score=11.1162
Thresh=0.0053, n=79, MSE=11.1789, Best Score=11.1162
Thresh=0.0059, n=78, MSE=11.1789, Best Score=11.1162
Thresh=0.0066, n=77, MSE=10.9953, Best Score=10.9953
Thresh=0.0068, n=76, MSE=11.0

#### ewma

In [5]:
fwd_data = get_data_ewma('FWD', FIELD_PLAYER_RELEVANT_FEATURES)
fwd_data = remove_correlated_features(fwd_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features_ewma.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

Num elements in FWD data before filter: 6967
Num elements in FWD data after filter: 2922
Found 7 pairs of highly correlated features.
Removed 6 features due to high correlation.
Original number of features; 21. Final number of features: 15
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 13 different feature thresholds...
Thresh=0.0590, n=13, MSE=11.6997, Best Score=11.6997
Thresh=0.0635, n=12, MSE=11.7078, Best Score=11.6997
Thresh=0.0636, n=11, MSE=11.5787, Best Score=11.5787
Thresh=0.0650, n=10, MSE=11.4178, Best Score=11.4178
Thresh=0.0666, n=9, MSE=11.7154, Best Score=11.4178
Thresh=0.0695, n=8, MSE=11.7346, Best Score=11.4178
Thresh=0.0753, n=7, MSE=11.8169, Best Score=11.4178
Thresh=0.0768, n=6, MSE=11.7111, Best Score=11.4178
Thresh=0.0770, n=5, MSE=11.6456, Best Score=11.4178
Thresh=0.0830, n=4, MSE=11.8166, Best Score=11.4178
Thresh=0.0955, n=3, MSE=12.2866, Best Score=11.4178
Thresh=0.1026, n=2, MSE=12.0689, Best Score=11.4178
Thre

#### Hybrid solution
Testing a solution using the ewma features as well as the top lagged features

In [9]:
fwd_data = get_data_hybrid('FWD', FIELD_PLAYER_RELEVANT_FEATURES)
fwd_data = remove_correlated_features(fwd_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features_hybrid.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

Creating hybrid feature set for position: FWD
Num elements in FWD data before filter: 6967
Num elements in FWD data after filter: 2922
Creating EWMA features...
--- Hybrid feature set created successfully ---
Found 34 pairs of highly correlated features.
Removed 24 features due to high correlation.
Original number of features; 65. Final number of features: 41
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 39 different feature thresholds...
Thresh=0.0000, n=39, MSE=11.7062, Best Score=11.7062
Thresh=0.0008, n=38, MSE=11.7062, Best Score=11.7062
Thresh=0.0088, n=37, MSE=11.7062, Best Score=11.7062
Thresh=0.0157, n=36, MSE=11.6560, Best Score=11.6560
Thresh=0.0161, n=35, MSE=11.4110, Best Score=11.4110
Thresh=0.0193, n=34, MSE=11.6782, Best Score=11.4110
Thresh=0.0199, n=33, MSE=11.6890, Best Score=11.4110
Thresh=0.0203, n=32, MSE=11.3624, Best Score=11.3624
Thresh=0.0209, n=31, MSE=11.3882, Best Score=11.3624
Thresh=0.0212, n=30, MSE=11.2963,

#### Variance 

In [7]:
fwd_data = get_data_hybrid_w_var('FWD', FIELD_PLAYER_RELEVANT_FEATURES)
fwd_data = remove_correlated_features(fwd_data, CORR_FEATURE_TRESHOLD, PRINT_INFO)

target_col = fwd_data['total_points']  # Extract the target column
fwd_data = fwd_data.drop(columns=['total_points'])  # Drop the target column from the features

fwd_features = perform_xgboost_selection(fwd_data, target_col)

# Store the selected features to a CSV file
pd.DataFrame(fwd_features).to_csv("../data/features/fwd_features_hybrid.csv", index=False, header=False)

# clear memory
del fwd_data, fwd_features, target_col

Creating hybrid feature set for position: FWD
Num elements in FWD data before filter: 6967
Num elements in FWD data after filter: 2922
Creating features for predicting the variance (volatility, uncertainty)...
Creating EWMA features...
Improved Hybrid feature set created successfully
Found 72 pairs of highly correlated features.
Removed 33 features due to high correlation.
Original number of features; 69. Final number of features: 36
Starting XGBoost Feature Selection
Training initial model to get feature importances
Testing 34 different feature thresholds...


c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Thresh=0.0020, n=34, MSE=11.3681, Best Score=11.3681


c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Thresh=0.0025, n=33, MSE=11.3681, Best Score=11.3681


...

Thresh=0.0439, n=4, MSE=13.2284, Best Score=11.1730


c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Thresh=0.0461, n=3, MSE=14.7250, Best Score=11.1730
Thresh=0.0515, n=2, MSE=10.9873, Best Score=10.9873


c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
c:\Users\trygt\fpl_dir\fpl_team_selector\.venv\lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


Thresh=0.0623, n=1, MSE=11.0622, Best Score=10.9873

Optimal number of features found: 2 (with MSE: 10.9873)

Selected features sorted by importance:
clean_sheets_lag1: 0.0623
clean_sheets_lag2: 0.0515


### MID

#### DEF

#### GK